In [1]:
import numpy as np
import pandas as pd


C:\Users\rajem\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
import pandas as pd
import numpy as np

# For reproducibility
np.random.seed(42)

# Number of samples
n_samples = 10000

# Garment information
garment_info = {
    "T-Shirt": {"base_time": 0.50, "ops": (5, 8)},
    "Shirt": {"base_time": 0.80, "ops": (8, 12)},
    "Trouser": {"base_time": 1.20, "ops": (10, 15)},
    "Jeans": {"base_time": 1.50, "ops": (12, 18)},
    "Kurti": {"base_time": 1.30, "ops": (10, 16)},
    "Dress": {"base_time": 1.80, "ops": (14, 20)},
    "Hoodie": {"base_time": 2.20, "ops": (16, 22)},
    "Jacket": {"base_time": 3.00, "ops": (20, 30)},
    "Blazer": {"base_time": 3.50, "ops": (22, 30)},
    "Uniform": {"base_time": 1.60, "ops": (12, 18)}
}

data = []

for _ in range(n_samples):

    garment = np.random.choice(list(garment_info.keys()))

    quantity = np.random.randint(100, 5001)

    stitch_ops = np.random.randint(
        garment_info[garment]["ops"][0],
        garment_info[garment]["ops"][1] + 1
    )

    base_time = garment_info[garment]["base_time"]

    # Complexity multiplier
    complexity = 1 + (stitch_ops / 50)

    # Random production efficiency
    efficiency = np.random.uniform(0.90, 1.10)

    # Random noise
    noise = np.random.normal(0, 30)

    sewing_time = (
        quantity
        * base_time
        * complexity
        * efficiency
    ) + noise

    sewing_time = round(max(sewing_time, 50), 2)

    data.append([
        garment,
        quantity,
        stitch_ops,
        sewing_time
    ])

df = pd.DataFrame(
    data,
    columns=[
        "garment_type",
        "order_quantity",
        "num_stitch_operations",
        "target_sewing_time_mins"
    ]
)

print(df.head())

print("\nDataset Shape:", df.shape)

df.to_csv("garment_sewing_dataset.csv", index=False)

print("\nDataset saved as garment_sewing_dataset.csv")

  garment_type  order_quantity  num_stitch_operations  target_sewing_time_mins
0       Hoodie             960                     22                  3149.04
1      Trouser            4526                     12                  7237.47
2        Jeans            3019                     14                  5223.27
3        Kurti            1284                     13                  2134.40
4       Blazer             574                     24                  2902.69

Dataset Shape: (10000, 4)

Dataset saved as garment_sewing_dataset.csv


In [3]:
import pandas as pd
df=pd.read_csv("garment_sewing_dataset.csv")

In [4]:
df

,garment_type,order_quantity,num_stitch_operations,target_sewing_time_mins
0,Hoodie,960,22,3149.04
1,Trouser,4526,12,7237.47
2,Jeans,3019,14,5223.27
3,Kurti,1284,13,2134.40
4,Blazer,574,24,2902.69
...,...,...,...,...
9995,T-Shirt,4906,8,2549.19
9996,Jeans,378,17,799.99
9997,Trouser,1428,11,1950.72
9998,Shirt,1657,8,1443.46


In [5]:
df.isnull().sum()

garment_type               0
order_quantity             0
num_stitch_operations      0
target_sewing_time_mins    0
dtype: int64

In [6]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
df["garment_type"] = encoder.fit_transform(df["garment_type"])


In [7]:
df["garment_type"]

0       2
1       8
2       4
3       5
4       0
       ..
9995    7
9996    4
9997    8
9998    6
9999    7
Name: garment_type, Length: 10000, dtype: int64

In [8]:
import joblib
joblib.dump(encoder, "encoder.pkl")

['encoder.pkl']

In [9]:
x=df.drop("target_sewing_time_mins",axis=1)

y = df["target_sewing_time_mins"]

In [10]:
x

,garment_type,order_quantity,num_stitch_operations
0,2,960,22
1,8,4526,12
2,4,3019,14
3,5,1284,13
4,0,574,24
...,...,...,...
9995,7,4906,8
9996,4,378,17
9997,8,1428,11
9998,6,1657,8


In [11]:
y

0       3149.04
1       7237.47
2       5223.27
3       2134.40
4       2902.69
         ...   
9995    2549.19
9996     799.99
9997    1950.72
9998    1443.46
9999     355.86
Name: target_sewing_time_mins, Length: 10000, dtype: float64

In [12]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [34]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=50,      # Reduce the number of trees
    max_depth=15,         # Limit tree depth
    min_samples_split=10,
    min_samples_leaf=5,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

In [35]:
model.fit(x_train, y_train)
pred=model.predict(x_test)

In [36]:
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

In [37]:
mae=mean_absolute_error(y_test,pred)
mse=mean_squared_error(y_test,pred)
r2=r2_score(y_test,pred)

In [38]:
print("mae:",mae)
print("mse:",mse)
print("r2:",r2)

mae: 359.95623747897713
mse: 305677.4444852035
r2: 0.9900939278950643


In [39]:
joblib.dump(model,"model1.pkl")
print("\n model saved successfully")


 model saved successfully


Garment Type	Encoded Value
Blazer	0
Dress	1
Hoodie	2
Jacket	3
Jeans	4
Kurti	5
Shirt	6
T-Shirt	7
Trouser	8
Uniform	9